# Spin-boson model: comparing three simulation approaches

This notebook reproduces the headline weak-coupling comparison from the thesis (Fig. 4.2): a single qubit coupled to an Ohmic bath, simulated three ways:

1. **Markov approximation** — analytical baseline, valid in the weak-coupling limit.
2. **Surrogate Hamiltonian** — N bath modes sampled near resonance, propagated with QuTiP.
3. **Chain mapping** — bath unitarily mapped to a 1D chain, propagated as an MPS via TEBD (TeNPy).

All three should agree at weak coupling. Strong coupling (set `alpha = 0.1`) breaks the Markov approximation while leaving chain mapping intact — that's the headline result.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import qutip as qt

from src.spin_boson import SpinBosonParams
from src.markov import evolve_markov
from src.surrogate import evolve_surrogate
from src.chain_mapping import evolve_chain_mapping

In [ ]:
# Weak coupling — Markov approximation should be valid.
p = SpinBosonParams(omega0=1.0, omega_c=2.0, alpha=0.001, s=1.0)
t_max = 20.0
psi_sys0 = qt.basis(2, 0)  # qubit in |up>
rho0 = psi_sys0 * psi_sys0.dag()

In [ ]:
result_markov = evolve_markov(p, t_max, rho0)

In [ ]:
result_surrogate, expect_surrogate, bath_freqs = evolve_surrogate(
    N=15, n_levels=2, p=p, t_max=t_max, psi_sys0=psi_sys0
)

In [ ]:
data_chain, _ = evolve_chain_mapping(L=100, p=p, t_max=t_max, chi_max=50)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5), dpi=120)
ax.plot(result_markov.times, result_markov.expect[2], 'k-', label='Markov approximation')
ax.plot(result_surrogate.times, expect_surrogate['sz'], label='Surrogate Hamiltonian')
ax.plot(data_chain['t'], data_chain['sz'][:, 0], '--', label='Chain mapping')
ax.set_xlabel('time')
ax.set_ylabel(r'$\langle \sigma_z \rangle$')
ax.set_xlim(0, t_max)
ax.grid(True, linestyle='--', linewidth=0.5)
ax.legend()
ax.set_title(f'alpha = {p.alpha} (weak coupling) — all three methods agree')
fig.tight_layout()